In [1]:
input_1 = 'file_input_uploads/Factordata-20241230-182848.zip'

In [2]:
!pip install openpyxl

import os
print(os.listdir("file_input_uploads"))


[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
['Factordata.zip']


In [3]:
import os
import openpyxl
import pandas as pd
import logging
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

class Config:
    def __init__(self):
        self.FOLDER_PATH = 'Factordata_2'
        if not os.path.exists(self.FOLDER_PATH):
            os.makedirs(self.FOLDER_PATH)
        
        self.WINDOWS = [12, 24]  # Quarterly, semester, annual, 2-year
        self.RANDOM_STATE = 42
        self.TEST_SIZE = 0.3
        self.N_ESTIMATORS = 200
        self.FACTOR_COLUMNS = [
            'Alpha..annualisiert.',
            'Value.Growth',
            'Small.Large',
            'Momentum',
            'Volatility'
        ]
        self.OUTPUT_DIR = os.path.join(self.FOLDER_PATH, 'analysis_output')
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)

class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()

    def load_and_preprocess(self, file_path):
        try:
            logger.info(f"Loading file: {os.path.basename(file_path)}")
            
            df = pd.read_excel(file_path)
            df.columns = df.columns.str.lower()
            if 'date' not in df.columns:
                raise ValueError("'date' column missing")
            
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df = df.dropna(subset=['date']).set_index('date')
            
            features, targets = [], []
            
            for col in self.config.FACTOR_COLUMNS:
                col_lower = col.lower()
                if col_lower not in df.columns:
                    logger.warning(f"Column {col} missing in {os.path.basename(file_path)}")
                    continue
                
                series = pd.to_numeric(df[col_lower], errors='coerce').ffill()
                features.append(series)
                
                for window in self.config.WINDOWS:
                    rolling_mean = series.rolling(window=window).mean()
                    rolling_std = series.rolling(window=window).std()
                    outliers = ((series - rolling_mean).abs() > 2 * rolling_std).astype(int)
                    targets.append(outliers)
            
            X = pd.concat(features, axis=1)
            X.columns = self.config.FACTOR_COLUMNS
            
            y = pd.concat(targets, axis=1)
            y.columns = [f"{col}_w{window}" for col in self.config.FACTOR_COLUMNS
                        for window in self.config.WINDOWS]
            
            logger.info(f"Processed file {file_path}: X shape {X.shape}, y shape {y.shape}")
            return X, y
            
        except Exception as e:
            logger.error(f"Error processing {file_path}: {e}")
            return None, None

class MultiOutputModelTrainer:
    def __init__(self, config):
        self.config = config
        self.models = {
            'random_forest': None,
            'svm': None,
            'logistic': None
        }
        self.feature_importance = {}
        self.valid_columns = None

    def prepare_data(self, X, y):
        X = X.fillna(X.mean())
        y = y.fillna(0)  # Fill NaN in target with 0 (no anomaly)

        # Check for single-class columns before split
        valid_columns = []
        for col in y.columns:
            if len(y[col].unique()) > 1:
                valid_columns.append(col)
            else:
                logger.warning(f"Dropping column {col} as it contains only one class")
        
        if not valid_columns:
            raise ValueError("No valid target columns found with more than one class")
            
        # Keep only valid columns
        y = y[valid_columns]
        self.valid_columns = valid_columns
        
        logger.info(f"Training with {len(valid_columns)} valid target columns")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.config.TEST_SIZE, random_state=self.config.RANDOM_STATE
        )

        logger.info(f"Prepared data: X_train shape {X_train.shape}, y_train shape {y_train.shape}")
        return X_train, X_test, y_train, y_test

    def train_model(self, X_train, y_train):
        logger.info("Training multiple models...")
        
        # Random Forest
        self.models['random_forest'] = MultiOutputClassifier(RandomForestClassifier(
            n_estimators=self.config.N_ESTIMATORS,
            random_state=self.config.RANDOM_STATE,
            class_weight='balanced'
        ))
        
        # SVM
        self.models['svm'] = MultiOutputClassifier(SVC(
            kernel='rbf',
            probability=True,
            random_state=self.config.RANDOM_STATE,
            class_weight='balanced'
        ))
        
        # Logistic Regression
        self.models['logistic'] = MultiOutputClassifier(LogisticRegression(
            random_state=self.config.RANDOM_STATE,
            class_weight='balanced',
            max_iter=1000
        ))

        # Train all models
        for model_name, model in self.models.items():
            logger.info(f"Training {model_name}...")
            try:
                model.fit(X_train, y_train)
                logger.info(f"Successfully trained {model_name}")
            except Exception as e:
                logger.error(f"Error training {model_name}: {e}")
                self.models[model_name] = None

    def evaluate_model(self, X_test, y_test):
        all_results = {}
        
        for model_name, model in self.models.items():
            if model is None:
                logger.warning(f"Skipping evaluation for {model_name} as it failed to train")
                continue
                
            logger.info(f"Evaluating {model_name}...")
            try:
                y_pred = model.predict(X_test)
                results = {}

                # Store feature importance for Random Forest
                if model_name == 'random_forest':
                    self.feature_importance = {}
                    for i, (estimator, target_name) in enumerate(zip(model.estimators_, 
                                                                   self.valid_columns)):
                        self.feature_importance[target_name] = dict(zip(
                            X_test.columns,
                            estimator.feature_importances_
                        ))

                # Calculate metrics for each target
                for i, col in enumerate(self.valid_columns):
                    try:
                        results[col] = {
                            'accuracy': accuracy_score(y_test.iloc[:, i], y_pred[:, i]),
                            'precision': precision_score(y_test.iloc[:, i], y_pred[:, i], 
                                                      zero_division=0),
                            'recall': recall_score(y_test.iloc[:, i], y_pred[:, i], 
                                                 zero_division=0),
                            'f1': f1_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
                        }
                    except Exception as e:
                        logger.error(f"Error calculating metrics for {col}: {e}")
                        results[col] = {
                            'accuracy': 0,
                            'precision': 0,
                            'recall': 0,
                            'f1': 0
                        }
                
                all_results[model_name] = results
                logger.info(f"{model_name} evaluation results: {results}")

                print(f"\n{model_name.upper()} Evaluation Results:")
                for key, value in results.items():
                    print(f"{key}: {value}")
            
            except Exception as e:
                logger.error(f"Error evaluating {model_name}: {e}")
                continue
                
        return all_results

def plot_feature_importance(trainer, config):
    plots_dir = os.path.join(config.OUTPUT_DIR, 'plots')
    os.makedirs(plots_dir, exist_ok=True)

    for target, importances in trainer.feature_importance.items():
        fig = px.bar(x=list(importances.keys()), y=list(importances.values()),
                    title=f'Feature Importance for {target}')
        fig.update_layout(xaxis_tickangle=-45)
        fig.write_html(os.path.join(plots_dir, f'feature_importance_{target}.html'))

def plot_performance_metrics(results, config):
    plots_dir = os.path.join(config.OUTPUT_DIR, 'plots')
    
    # Create comparison plots for each metric
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        fig = go.Figure()
        
        for model_name, model_results in results.items():
            metrics_df = pd.DataFrame(model_results).transpose()
            fig.add_trace(go.Bar(
                name=model_name,
                x=metrics_df.index,
                y=metrics_df[metric]
            ))
            
        fig.update_layout(
            title=f'{metric.capitalize()} Comparison Across Models',
            barmode='group',
            xaxis_tickangle=-90
        )
        fig.write_html(os.path.join(plots_dir, f'comparison_{metric}.html'))
    
    # Save detailed results to CSV
    comparison_df = pd.concat({
        model_name: pd.DataFrame(model_results).transpose()
        for model_name, model_results in results.items()
    }, axis=1)
    
    comparison_df.to_csv(os.path.join(config.OUTPUT_DIR, 'model_comparison.csv'))
    return comparison_df

def main():
    config = Config()
    preprocessor = DataPreprocessor(config)
    trainer = MultiOutputModelTrainer(config)

    logger.info("Starting enhanced anomaly detection analysis...")

    all_features, all_targets = [], []

    for filename in os.listdir(config.FOLDER_PATH):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(config.FOLDER_PATH, filename)
            X, y = preprocessor.load_and_preprocess(file_path)
            if X is not None and y is not None:
                all_features.append(X)
                all_targets.append(y)
                
                # Save the anomaly labels (y) for this dataset
                y.to_csv(os.path.join(config.OUTPUT_DIR, f"anomaly_labels_{filename}.csv"))

    if not all_features or not all_targets:
        logger.error("No valid data found for processing")
        return

    X = pd.concat(all_features)
    y = pd.concat(all_targets)

    X_train, X_test, y_train, y_test = trainer.prepare_data(X, y)
    trainer.train_model(X_train, y_train)
    results = trainer.evaluate_model(X_test, y_test)

    plot_feature_importance(trainer, config)
    metrics_df = plot_performance_metrics(results, config)

    # Save all results
    for model_name, model_results in results.items():
        pd.DataFrame(model_results).to_csv(
            os.path.join(config.OUTPUT_DIR, f'evaluation_results_{model_name}.csv'))
    
    pd.DataFrame(trainer.feature_importance).transpose().to_csv(
        os.path.join(config.OUTPUT_DIR, 'feature_importance.csv'))

    logger.info("Analysis and visualizations completed successfully")

if __name__ == "__main__":
    main()

2025-01-03 09:14:10,693 - INFO - Starting enhanced anomaly detection analysis...
2025-01-03 09:14:12,421 - INFO - Loading file: 21216.xlsx
2025-01-03 09:14:12,593 - INFO - Processed file Factordata_2/21216.xlsx: X shape (91, 5), y shape (91, 10)
2025-01-03 09:14:12,902 - INFO - Loading file: 3645.xlsx
2025-01-03 09:14:13,144 - INFO - Processed file Factordata_2/3645.xlsx: X shape (90, 5), y shape (90, 10)
2025-01-03 09:14:13,361 - INFO - Loading file: Finreon.xlsx
2025-01-03 09:14:13,569 - INFO - Processed file Factordata_2/Finreon.xlsx: X shape (92, 5), y shape (92, 10)
2025-01-03 09:14:13,849 - INFO - Loading file: GAM.xlsx
2025-01-03 09:14:14,038 - INFO - Processed file Factordata_2/GAM.xlsx: X shape (91, 5), y shape (91, 10)
2025-01-03 09:14:14,294 - INFO - Loading file: IAM.xlsx
2025-01-03 09:14:14,473 - INFO - Processed file Factordata_2/IAM.xlsx: X shape (91, 5), y shape (91, 10)
2025-01-03 09:14:14,800 - INFO - Loading file: Lo.xlsx
2025-01-03 09:14:15,007 - INFO - Processed fi

In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Simple configuration class
class Config:
    def __init__(self):
        self.RANDOM_STATE = 42
        self.N_ESTIMATORS = 200
        self.FACTOR_COLUMNS = ['Alpha', 'Value', 'Small', 'Momentum', 'Volatility']

# Main trainer class with simplified functionality
class MultiOutputModelTrainer:
    def __init__(self, n_splits=5):
        self.n_splits = n_splits
        self.random_state = 42
        
    def perform_cross_validation(self, X, y):
        print("\nStarting cross-validation...")
        
        # Initialize KFold
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        
        # Initialize models
        models = {
            'random_forest': MultiOutputClassifier(RandomForestClassifier(random_state=self.random_state)),
            'svm': MultiOutputClassifier(SVC(probability=True, random_state=self.random_state)),
            'logistic': MultiOutputClassifier(LogisticRegression(random_state=self.random_state, max_iter=1000))
        }
        
        results = {}
        
        # For each model
        for model_name, model in models.items():
            print(f"\nEvaluating {model_name.upper()}...")
            fold_scores = []
            
            # For each fold
            for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
                
                print(f"Processing fold {fold}/{self.n_splits}")
                
                # Train model
                model.fit(X_train, y_train)
                y_pred = model.predict(X_val)
                
                # Calculate metrics
                fold_metrics = {}
                for i, col in enumerate(y.columns):
                    metrics = {
                        'accuracy': accuracy_score(y_val.iloc[:, i], y_pred[:, i]),
                        'precision': precision_score(y_val.iloc[:, i], y_pred[:, i], zero_division=0),
                        'recall': recall_score(y_val.iloc[:, i], y_pred[:, i], zero_division=0),
                        'f1': f1_score(y_val.iloc[:, i], y_pred[:, i], zero_division=0)
                    }
                    fold_metrics[col] = metrics
                
                fold_scores.append(fold_metrics)
                
                # Print fold results
                print(f"\nFold {fold} Results:")
                for col, metrics in fold_metrics.items():
                    print(f"\nTarget: {col}")
                    for metric, value in metrics.items():
                        print(f"{metric}: {value:.3f}")
            
            # Calculate and store average results
            avg_results = {}
            for col in y.columns:
                avg_results[col] = {
                    metric: np.mean([fold[col][metric] for fold in fold_scores])
                    for metric in ['accuracy', 'precision', 'recall', 'f1']
                }
            
            results[model_name] = {
                'fold_results': fold_scores,
                'average_results': avg_results
            }
            
            # Print average results for this model
            print(f"\n{model_name.upper()} - Average Results:")
            for col, metrics in avg_results.items():
                print(f"\nTarget: {col}")
                for metric, value in metrics.items():
                    print(f"{metric}: {value:.3f}")
        
        return results

# Main execution
def main():
    # Generate sample data
    np.random.seed(42)
    n_samples = 1000
    n_features = 5
    
    # Create features
    X = pd.DataFrame(
        np.random.randn(n_samples, n_features),
        columns=['Alpha', 'Value', 'Small', 'Momentum', 'Volatility']
    )
    
    # Create target variables with some patterns
    y_data = {}
    windows = [6, 12, 24]
    for col in X.columns:
        for window in windows:
            target = ((X[col] > 0.5).astype(int) + 
                     (np.random.randn(n_samples) > 1).astype(int)) > 0
            y_data[f"{col}_w{window}"] = target.astype(int)
    
    y = pd.DataFrame(y_data)
    
    # Perform cross-validation
    trainer = MultiOutputModelTrainer(n_splits=5)
    results = trainer.perform_cross_validation(X, y)
    
    # Print final summary
    print("\n" + "="*50)
    print("FINAL SUMMARY")
    print("="*50)
    
    for model_name, model_results in results.items():
        print(f"\n{model_name.upper()} OVERALL PERFORMANCE:")
        avg_metrics = {metric: [] for metric in ['accuracy', 'precision', 'recall', 'f1']}
        
        for target, metrics in model_results['average_results'].items():
            for metric, value in metrics.items():
                avg_metrics[metric].append(value)
        
        for metric, values in avg_metrics.items():
            mean_value = np.mean(values)
            std_value = np.std(values)
            print(f"{metric:9}: {mean_value:.3f} ± {std_value:.3f}")

if __name__ == "__main__":
    main()

Target: Small_w12
accuracy: 0.860
precision: 0.910
recall: 0.735
f1: 0.813

Target: Small_w24
accuracy: 0.865
precision: 0.938
recall: 0.726
f1: 0.819

Target: Momentum_w6
accuracy: 0.825
precision: 0.930
recall: 0.688
f1: 0.790

Target: Momentum_w12
accuracy: 0.850
precision: 0.873
recall: 0.775
f1: 0.821

Target: Momentum_w24
accuracy: 0.870
precision: 0.905
recall: 0.779
f1: 0.838

Target: Volatility_w6
accuracy: 0.815
precision: 0.846
recall: 0.725
f1: 0.781

Target: Volatility_w12
accuracy: 0.890
precision: 0.932
recall: 0.802
f1: 0.863

Target: Volatility_w24
accuracy: 0.825
precision: 0.893
recall: 0.713
f1: 0.793
Processing fold 3/5

Fold 3 Results:

Target: Alpha_w6
accuracy: 0.865
precision: 0.944
recall: 0.680
f1: 0.791

Target: Alpha_w12
accuracy: 0.840
precision: 0.881
recall: 0.675
f1: 0.765

Target: Alpha_w24
accuracy: 0.795
precision: 0.902
recall: 0.611
f1: 0.728

Target: Value_w6
accuracy: 0.865
precision: 0.929
recall: 0.747
f1: 0.828

Target: Value_w12
accuracy: 0.7

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(asctime)s - %(message)s')
logger = logging.getLogger()

class Config:
    def __init__(self):
        self.RANDOM_STATE = 42
        self.N_ESTIMATORS = 200
        self.WINDOWS = [12, 24]
        self.START_DATE = '2014-01-01'
        self.END_DATE = '2024-12-31'
        self.FOLDER_PATH = 'Factordata_2'
        self.FACTOR_COLUMNS = [
            'Alpha..annualisiert.',
            'Value.Growth',
            'Small.Large',
            'Momentum',
            'Volatility'
        ]

        if not os.path.exists(self.FOLDER_PATH):
            os.makedirs(self.FOLDER_PATH)

class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()
        self.imputer = SimpleImputer(strategy='mean')

    def load_and_preprocess_folder(self):
        all_features = []
        all_targets = []

        for file_name in os.listdir(self.config.FOLDER_PATH):
            if file_name.endswith('.xlsx'):
                file_path = os.path.join(self.config.FOLDER_PATH, file_name)
                logger.info(f"Processing file: {file_name}")

                try:
                    data = pd.read_excel(file_path)
                    data.columns = data.columns.str.strip()

                    if 'Date' not in data.columns:
                        logger.warning(f"File {file_name} missing 'Date' column, skipping.")
                        continue

                    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
                    data = data.dropna(subset=['Date']).set_index('Date')

                    features, targets = self.process_data(data)
                    if features is not None and targets is not None:
                        all_features.append(features)
                        all_targets.append(targets)

                except Exception as e:
                    logger.error(f"Error processing file {file_name}: {e}")

        if not all_features or not all_targets:
            raise ValueError("No valid data found in the folder.")

        final_features = pd.concat(all_features)
        final_targets = pd.concat(all_targets)

        return final_features, final_targets

    def process_data(self, data):
        try:
            features = []
            targets = []

            for col in self.config.FACTOR_COLUMNS:
                col_lower = col.lower()
                if col_lower not in data.columns.str.lower():
                    logger.warning(f"Column {col} missing, skipping.")
                    continue

                series = pd.to_numeric(data[col], errors='coerce').ffill()
                features.append(series)

                for window in self.config.WINDOWS:
                    rolling_mean = series.rolling(window=window, min_periods=1).mean()
                    rolling_std = series.rolling(window=window, min_periods=1).std()
                    outliers = ((series - rolling_mean).abs() > 2 * rolling_std).astype(int)
                    targets.append(outliers)

            if not features or not targets:
                return None, None

            X = pd.concat(features, axis=1)
            X.columns = self.config.FACTOR_COLUMNS[:len(features)]

            y = pd.concat(targets, axis=1)
            y.columns = [f"{col}_w{window}" for col in self.config.FACTOR_COLUMNS[:len(features)] 
                         for window in self.config.WINDOWS]

            return X, y

        except Exception as e:
            logger.error(f"Error in process_data: {e}")
            return None, None

class ModelTrainer:
    def __init__(self, config):
        self.config = config

    def prepare_data(self, X, y):
        try:
            X = X.fillna(X.mean())
            y = y.fillna(0)

            valid_columns = []
            for col in y.columns:
                if len(y[col].unique()) > 1:
                    valid_columns.append(col)
                else:
                    logger.warning(f"Dropping column {col} with only one class.")

            if not valid_columns:
                raise ValueError("No valid target columns found with more than one class.")

            y = y[valid_columns]
            return X, y

        except Exception as e:
            logger.error(f"Error in prepare_data: {e}")
            raise

    def cross_validate(self, X, y, n_splits=5):
        try:
            models = {
                'random_forest': MultiOutputClassifier(RandomForestClassifier(
                    n_estimators=self.config.N_ESTIMATORS, 
                    random_state=self.config.RANDOM_STATE
                )),
                'svm': MultiOutputClassifier(SVC(probability=True, random_state=self.config.RANDOM_STATE)),
                'logistic': MultiOutputClassifier(LogisticRegression(random_state=self.config.RANDOM_STATE, max_iter=1000))
            }

            kf = KFold(n_splits=n_splits, shuffle=True, random_state=self.config.RANDOM_STATE)
            results = {}

            for model_name, model in models.items():
                logger.info(f"Starting cross-validation for {model_name}.")
                model_results = {}

                for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
                    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_val)

                    fold_metrics = {}
                    for target_idx, target_col in enumerate(y.columns):
                        metrics = {
                            'accuracy': accuracy_score(y_val.iloc[:, target_idx], y_pred[:, target_idx]),
                            'precision': precision_score(y_val.iloc[:, target_idx], y_pred[:, target_idx], zero_division=0),
                            'recall': recall_score(y_val.iloc[:, target_idx], y_pred[:, target_idx], zero_division=0),
                            'f1': f1_score(y_val.iloc[:, target_idx], y_pred[:, target_idx], zero_division=0)
                        }
                        fold_metrics[target_col] = metrics

                    for target_col, metrics in fold_metrics.items():
                        if target_col not in model_results:
                            model_results[target_col] = []
                        model_results[target_col].append(metrics)

                avg_results = {
                    target_col: {
                        metric: np.mean([fold[metric] for fold in folds])
                        for metric in folds[0]
                    }
                    for target_col, folds in model_results.items()
                }

                results[model_name] = avg_results

            for model_name, metrics in results.items():
                print(f"\n{model_name.upper()} Evaluation Results:")
                for target, target_metrics in metrics.items():
                    print(f"{target}: {target_metrics}")

        except Exception as e:
            logger.error(f"Error in cross-validation: {e}")
            raise

def main():
    config = Config()
    preprocessor = DataPreprocessor(config)
    trainer = ModelTrainer(config)

    X, y = preprocessor.load_and_preprocess_folder()
    X, y = trainer.prepare_data(X, y)

    if X.empty or y.empty:
        logger.error("No valid data for modeling.")
        return

    trainer.cross_validate(X, y, n_splits=5)

if __name__ == "__main__":
    main()


2025-01-03 09:14:48,867 - INFO - Processing file: 21216.xlsx
2025-01-03 09:14:49,056 - INFO - Processing file: 3645.xlsx
2025-01-03 09:14:49,232 - INFO - Processing file: Finreon.xlsx
2025-01-03 09:14:49,421 - INFO - Processing file: GAM.xlsx
2025-01-03 09:14:49,589 - INFO - Processing file: IAM.xlsx
2025-01-03 09:14:49,788 - INFO - Processing file: Lo.xlsx
2025-01-03 09:14:49,958 - INFO - Processing file: Pictet.xlsx
2025-01-03 09:14:50,135 - INFO - Processing file: SGKB.xlsx
2025-01-03 09:14:50,306 - INFO - Processing file: SaraSelect.xlsx
2025-01-03 09:14:50,490 - INFO - Processing file: Vontobel.xlsx
2025-01-03 09:14:50,661 - INFO - Processing file: creditsuisse.xlsx
2025-01-03 09:14:50,859 - INFO - Processing file: zCapital.xlsx
2025-01-03 09:14:51,035 - INFO - Starting cross-validation for random_forest.
2025-01-03 09:15:09,732 - INFO - Starting cross-validation for svm.
2025-01-03 09:15:12,748 - INFO - Starting cross-validation for logistic.

RANDOM_FOREST Evaluation Results:
Al

In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(asctime)s - %(message)s')
logger = logging.getLogger()

class Config:
    def __init__(self):
        self.RANDOM_STATE = 42
        self.N_ESTIMATORS = 50
        self.WINDOWS = [6, 12, 24]
        self.START_DATE = '2014-01-01'
        self.END_DATE = '2024-12-31'
        self.FOLDER_PATH = 'Factordata_2'
        self.FACTOR_COLUMNS = [
            'Alpha..annualisiert.',
            'Value.Growth',
            'Small.Large',
            'Momentum',
            'Volatility'
        ]

        if not os.path.exists(self.FOLDER_PATH):
            os.makedirs(self.FOLDER_PATH)

class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()
        self.imputer = SimpleImputer(strategy='mean')

    def load_and_preprocess_folder(self):
        all_features = []
        all_targets = []

        for file_name in os.listdir(self.config.FOLDER_PATH):
            if file_name.endswith('.xlsx'):
                file_path = os.path.join(self.config.FOLDER_PATH, file_name)
                logger.info(f"Processing file: {file_name}")

                try:
                    data = pd.read_excel(file_path)
                    data.columns = data.columns.str.strip()

                    if 'Date' not in data.columns:
                        logger.warning(f"File {file_name} missing 'Date' column, skipping.")
                        continue

                    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
                    data = data.dropna(subset=['Date']).set_index('Date')

                    features, targets = self.process_data(data)
                    if features is not None and targets is not None:
                        all_features.append(features)
                        all_targets.append(targets)

                except Exception as e:
                    logger.error(f"Error processing file {file_name}: {e}")

        if not all_features or not all_targets:
            raise ValueError("No valid data found in the folder.")

        final_features = pd.concat(all_features)
        final_targets = pd.concat(all_targets)

        return final_features, final_targets

    def process_data(self, data):
        try:
            features = []
            targets = []

            for col in self.config.FACTOR_COLUMNS:
                col_lower = col.lower()
                if col_lower not in data.columns.str.lower():
                    logger.warning(f"Column {col} missing, skipping.")
                    continue

                series = pd.to_numeric(data[col], errors='coerce').ffill()
                features.append(series)

                for window in self.config.WINDOWS:
                    rolling_mean = series.rolling(window=window, min_periods=1).mean()
                    rolling_std = series.rolling(window=window, min_periods=1).std()
                    outliers = ((series - rolling_mean).abs() > 2 * rolling_std).astype(int)
                    targets.append(outliers)

            if not features or not targets:
                return None, None

            X = pd.concat(features, axis=1)
            X.columns = self.config.FACTOR_COLUMNS[:len(features)]

            y = pd.concat(targets, axis=1)
            y.columns = [f"{col}_w{window}" for col in self.config.FACTOR_COLUMNS[:len(features)] 
                         for window in self.config.WINDOWS]

            return X, y

        except Exception as e:
            logger.error(f"Error in process_data: {e}")
            return None, None

class ModelTrainer:
    def __init__(self, config):
        self.config = config

    def prepare_data(self, X, y):
        try:
            X = X.fillna(X.mean())
            y = y.fillna(0)

            valid_columns = []
            for col in y.columns:
                if len(y[col].unique()) > 1:
                    valid_columns.append(col)
                else:
                    logger.warning(f"Dropping column {col} with only one class.")

            if not valid_columns:
                raise ValueError("No valid target columns found with more than one class.")

            y = y[valid_columns]
            return X, y

        except Exception as e:
            logger.error(f"Error in prepare_data: {e}")
            raise

    def cross_validate(self, X, y, n_splits=5):
        try:
            models = {
                'random_forest': MultiOutputClassifier(RandomForestClassifier(
                    n_estimators=self.config.N_ESTIMATORS, 
                    random_state=self.config.RANDOM_STATE
                )),
                'svm': MultiOutputClassifier(SVC(probability=True, random_state=self.config.RANDOM_STATE)),
                'logistic': MultiOutputClassifier(LogisticRegression(random_state=self.config.RANDOM_STATE, max_iter=1000))
            }

            kf = KFold(n_splits=n_splits, shuffle=True, random_state=self.config.RANDOM_STATE)
            results = {}

            for model_name, model in models.items():
                logger.info(f"Starting cross-validation for {model_name}.")
                fold_results = []

                for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
                    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_val)

                    metrics = {
                        'accuracy': accuracy_score(y_val, y_pred),
                        'precision': precision_score(y_val, y_pred, average='macro', zero_division=0),
                        'recall': recall_score(y_val, y_pred, average='macro', zero_division=0),
                        'f1': f1_score(y_val, y_pred, average='macro', zero_division=0)
                    }

                    fold_results.append(metrics)
                    logger.info(f"Fold {fold} results for {model_name}: {metrics}")

                results[model_name] = fold_results

            logger.info("Cross-validation completed for all models.")
            for model_name, folds in results.items():
                avg_metrics = {metric: np.mean([fold[metric] for fold in folds]) for metric in folds[0]}
                logger.info(f"{model_name} average metrics: {avg_metrics}")

        except Exception as e:
            logger.error(f"Error in cross-validation: {e}")
            raise

def main():
    config = Config()
    preprocessor = DataPreprocessor(config)
    trainer = ModelTrainer(config)

    X, y = preprocessor.load_and_preprocess_folder()
    X, y = trainer.prepare_data(X, y)

    if X.empty or y.empty:
        logger.error("No valid data for modeling.")
        return

    trainer.cross_validate(X, y, n_splits=5)

if __name__ == "__main__":
    main()


2025-01-03 09:15:15,161 - INFO - Processing file: 21216.xlsx
2025-01-03 09:15:15,385 - INFO - Processing file: 3645.xlsx
2025-01-03 09:15:15,579 - INFO - Processing file: Finreon.xlsx
2025-01-03 09:15:15,764 - INFO - Processing file: GAM.xlsx
2025-01-03 09:15:15,937 - INFO - Processing file: IAM.xlsx
2025-01-03 09:15:16,103 - INFO - Processing file: Lo.xlsx
2025-01-03 09:15:16,290 - INFO - Processing file: Pictet.xlsx
2025-01-03 09:15:16,494 - INFO - Processing file: SGKB.xlsx
2025-01-03 09:15:16,667 - INFO - Processing file: SaraSelect.xlsx
2025-01-03 09:15:16,839 - INFO - Processing file: Vontobel.xlsx
2025-01-03 09:15:17,032 - INFO - Processing file: creditsuisse.xlsx
2025-01-03 09:15:17,209 - INFO - Processing file: zCapital.xlsx
2025-01-03 09:15:17,395 - INFO - Starting cross-validation for random_forest.
2025-01-03 09:15:18,657 - INFO - Fold 1 results for random_forest: {'accuracy': 0.5799086757990868, 'precision': 0.31103174603174605, 'recall': 0.07039355965671755, 'f1': 0.11158

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b0d79e99-778c-4964-b163-e34d369ad413' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>